# 📈 Chapter 11: General Linear Models and Least Squares

---

## 11.1 Introduction: The Crown Jewel of Applied Linear Algebra

If you want to extract relationships, predict future outcomes, or understand the influence of variables within a dataset, you need to fit a statistical model. The **General Linear Model (GLM)** is an incredibly flexible framework that encompasses linear regression, multiple regression, ANOVA, t-tests, and many other statistical methods.

In abstract linear algebra, we try to solve the system $\mathbf{A}\mathbf{x} = \mathbf{b}$. 
In data science and statistics, we rewrite this exact same equation using different letters to represent our models:
$$ \mathbf{X}\beta = \mathbf{y} $$

In this chapter, you will learn how to set up this equation, why an exact solution is almost never possible with real-world data, and how we use the **Least Squares** algorithm (powered by the Left-Inverse and QR Decomposition) to find the best possible approximation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

# Set plotting style
%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 11.")

## 11.2 Terminology and Setting Up a GLM

Let's define the components of the equation $\mathbf{X}\beta = \mathbf{y}$:

1. **$\mathbf{X}$ (The Design Matrix):** This matrix contains our independent variables, features, or predictors. Each column is a specific feature (e.g., age, weight, hours studied), and each row is an observation. 
    * *Crucial Note:* We almost always add a column of strictly `1`s to this matrix. This is called the **Intercept** column, which accounts for the baseline value of the data.
2. **$\mathbf{y}$ (The Target Vector):** This is the dependent variable, or the outcome we are trying to predict (e.g., test scores, blood pressure, housing prices).
3. **$\beta$ (The Coefficients/Weights):** This is the vector of unknown parameters we want to find. It tells us how much each column in $\mathbf{X}$ contributes to explaining $\mathbf{y}$.

Because we almost always have more observations (rows) than features (columns), $\mathbf{X}$ is a **tall matrix**.

In [ ]:
# Let's set up a simple General Linear Model (Simple Linear Regression)
# Predicting a person's income (y) based on their years of education (feature 1)

num_observations = 50

# Feature 1: Years of education (random numbers between 10 and 20)
education = np.random.uniform(10, 20, num_observations)

# Target y: Income (in thousands), linearly related to education plus some random noise
income = 2.5 * education + 15 + np.random.normal(0, 4, num_observations)

# Constructing the Design Matrix (X)
# It needs two columns: The intercept (all 1s) and the education data
intercept = np.ones(num_observations)
X = np.column_stack((intercept, education))

print(f"Shape of Design Matrix X: {X.shape} (Tall Matrix)")
print(f"Shape of Target Vector y: {income.shape}")
print(f"We are looking for a beta vector of shape: ({X.shape[1]}, )")

## 11.3 Solving GLMs: The Least Squares Formula

We want to solve $\mathbf{X}\beta = \mathbf{y}$. 
To isolate $\beta$, we need to "divide" by $\mathbf{X}$. However, $\mathbf{X}$ is a tall rectangular matrix, meaning it has no standard inverse. 

To solve this, we use the **Left-Inverse** we derived in Chapter 8. By multiplying both sides by $\mathbf{X}^T$, we create a square, invertible matrix $(\mathbf{X}^T\mathbf{X})$:

$$ \mathbf{X}^T\mathbf{X}\beta = \mathbf{X}^T\mathbf{y} $$
$$ (\mathbf{X}^T\mathbf{X})^{-1}(\mathbf{X}^T\mathbf{X})\beta = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y} $$
$$ \beta = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y} $$

This beautiful equation is the **Ordinary Least Squares (OLS)** solution. It finds the specific $\beta$ weights that minimize the squared error between our predictions and the real data.

In [ ]:
# 1. Solving manually using the Left-Inverse equation
XtX = X.T @ X
XtX_inv = np.linalg.inv(XtX)
beta_manual = XtX_inv @ X.T @ income

# 2. Solving using NumPy's built-in Least Squares function
# np.linalg.lstsq returns the solution, residuals, rank, and singular values
beta_numpy, residuals, rank, s = np.linalg.lstsq(X, income, rcond=None)

print(f"Manual Beta   : Intercept = {beta_manual[0]:.2f}, Slope = {beta_manual[1]:.2f}")
print(f"NumPy Beta    : Intercept = {beta_numpy[0]:.2f}, Slope = {beta_numpy[1]:.2f}")

# Visualization of the Model Fit
plt.figure(figsize=(7, 5))
plt.scatter(education, income, color='teal', label='Raw Data (y)')

# Generating the regression line using our calculated betas
x_range = np.linspace(10, 20, 10)
y_pred = beta_manual[0] + beta_manual[1] * x_range
plt.plot(x_range, y_pred, color='red', linewidth=3, label='Least Squares Fit')

plt.xlabel("Years of Education")
plt.ylabel("Income ($1000s)")
plt.title("General Linear Model via Least Squares")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11.4 Is the Solution Exact? (Residuals)

In pure mathematics, if $Ax = b$, then calculating $Ax$ gives you exactly $b$. 
In data science, $\mathbf{X}\beta$ almost never equals $\mathbf{y}$ exactly because real-world data contains noise. Instead, $\mathbf{X}\beta$ gives us $\mathbf{\hat{y}}$ (read as "y-hat"), which are our **predicted values**.

The difference between the actual data and our predictions is called the **Residuals** (or errors), denoted by $\epsilon$:
$$ \epsilon = \mathbf{y} - \mathbf{\hat{y}} $$

The least squares algorithm mathematically guarantees that the sum of the squared residuals $(\epsilon^T\epsilon)$ is as small as theoretically possible.

In [ ]:
# Calculate the predicted values (y_hat)
y_hat = X @ beta_manual

# Calculate the residuals (epsilon)
residuals = income - y_hat

print(f"Sum of squared residuals (Error): {np.dot(residuals, residuals):.2f}")
print("Any other beta vector would result in a larger error!")

# An amazing property of Least Squares: The residuals are ORTHOGONAL to the design matrix
# That means X.T @ residuals should be perfectly zero (or machine precision zero)
orthogonality_check = X.T @ residuals
print("\nOrthogonality check (X^T @ residuals):")
print(np.round(orthogonality_check, 4))

## 11.5 A Geometric Perspective on Least Squares

Let's re-interpret everything we just did using the geometric Matrix Spaces we learned in Chapter 6.

1. Our Design Matrix $\mathbf{X}$ has a **Column Space** $C(\mathbf{X})$. Any vector that can be perfectly predicted by our model MUST live entirely flat on this column space.
2. Our target data $\mathbf{y}$ is a vector floating somewhere out in the universe. Because of noise, $\mathbf{y}$ points outside of the column space $C(\mathbf{X})$. This is why an exact solution is impossible.
3. To get the best approximation, we need to find a point on the column space that is geographically as close to $\mathbf{y}$ as possible. 
4. The shortest distance from a point to a plane is a line that drops straight down at a **perfect 90-degree angle (Orthogonal)**. 

Therefore, Least Squares is simply the process of **Orthogonally Projecting** the target vector $\mathbf{y}$ onto the column space of $\mathbf{X}$. The shadow it casts on the plane is $\mathbf{\hat{y}}$, and the distance it dropped is the residual $\epsilon$!

## 11.6 Least Squares via QR Decomposition

Remember in Chapter 9 when I said you should never explicitly compute the matrix inverse $(\mathbf{X}^T\mathbf{X})^{-1}$ because it is numerically unstable? 

Squaring a matrix (multiplying it by its transpose) drastically increases its Condition Number, amplifying floating-point rounding errors. If our features are highly correlated (multicollinearity), the inverse calculation might literally blow up the computer's memory limits.

Instead, professional software computes Least Squares using the **QR Decomposition**. 
Let's substitute $\mathbf{X}$ with $\mathbf{QR}$ in the GLM equation:

$$ \mathbf{X}\beta = \mathbf{y} $$
$$ \mathbf{QR}\beta = \mathbf{y} $$

Pre-multiply by $\mathbf{Q}^T$ (which is $\mathbf{Q}^{-1}$ because $\mathbf{Q}$ is orthogonal):
$$ \mathbf{Q}^T\mathbf{QR}\beta = \mathbf{Q}^T\mathbf{y} $$
$$ \mathbf{I}\mathbf{R}\beta = \mathbf{Q}^T\mathbf{y} $$
$$ \mathbf{R}\beta = \mathbf{Q}^T\mathbf{y} $$

Because $\mathbf{R}$ is an upper-triangular matrix, we don't even need an inverse! We can just use simple algebraic back-substitution to solve for $\beta$. This is incredibly stable and fast.

In [ ]:
# 1. Decompose the Design Matrix X into Q and R
Q, R = np.linalg.qr(X)

# 2. Compute the right-hand side of the equation (Q^T @ y)
qty = Q.T @ income

# 3. Solve for beta using back-substitution (scipy's solve_triangular)
# We tell the solver that the matrix is upper-triangular for maximum efficiency
beta_qr = la.solve_triangular(R, qty)

print(f"Beta from Left-Inverse (Unstable) : {beta_manual}")
print(f"Beta from QR Decomposition (Safe) : {beta_qr}")
print("\nNotice that the answers are identical, but the QR method avoided matrix inversion!")

## 11.7 Chapter Summary

This chapter brought together almost everything we have learned so far:
- **The General Linear Model (GLM)** expresses statistical relationships as a matrix equation $\mathbf{X}\beta = \mathbf{y}$.
- Because real data contains noise, $y$ is never perfectly inside the column space of $X$, making an exact solution impossible.
- The **Least Squares** algorithm finds the optimal weights $\beta$ by minimizing the sum of the squared residuals.
- Geometrically, this means taking the vector $\mathbf{y}$ and projecting it orthogonally onto the column space of $\mathbf{X}$.
- To guarantee numerical stability, we avoid the explicit left-inverse $(\mathbf{X}^T\mathbf{X})^{-1}$ and instead use the **QR Decomposition** to safely and efficiently solve for $\beta$.